# Data Quality — Setup Notebook

**Run this notebook once** to create the two tables that power the validation system.

- `check_registry` — stores the model names and DAX expressions you want to validate
- `validation_results` — stores the results of each run (written by the job)

After this setup, go to `data_quality_add_checks_notebook` to add your first checks.

In [ ]:
# ── Step 1: Configuration ────────────────────────────────────────────────────
LAKEHOUSE_NAME = "MyLakehouse"    # Your existing Lakehouse name
SCHEMA_NAME    = "data_quality"   # Schema (namespace) to create inside that Lakehouse

FULL_SCHEMA = f"{LAKEHOUSE_NAME}.{SCHEMA_NAME}"

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {FULL_SCHEMA}")

# --- check_registry: the table business users fill in ---
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {FULL_SCHEMA}.check_registry (
    check_name      STRING  NOT NULL COMMENT 'Same name across all models so the job knows to compare them',
    model_name      STRING  NOT NULL COMMENT 'Friendly label — any name you choose',
    workspace_name  STRING  NOT NULL COMMENT 'Fabric workspace that contains the semantic model',
    dataset_name    STRING  NOT NULL COMMENT 'Semantic model / dataset name',
    dax_expression  STRING  NOT NULL COMMENT 'DAX returning a single number, e.g. EVALUATE ROW(\"value\", [Sales Amount])',
    run_frequency   STRING  NOT NULL DEFAULT 'ONCE_PER_DAY' COMMENT 'ONCE_PER_DAY (run max once per day) or MULTIPLE_PER_DAY (can run multiple times)',
    is_active       BOOLEAN NOT NULL DEFAULT true COMMENT 'Set to false to skip this row without deleting it'
)
USING DELTA
COMMENT 'Add one row per model per check — all rows sharing a check_name will be compared'
""")

# --- validation_results: written by the job, read-only for users ---
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {FULL_SCHEMA}.validation_results (
    run_id          STRING    NOT NULL COMMENT 'UUID identifying this specific run',
    run_date        DATE      NOT NULL,
    run_timestamp   TIMESTAMP NOT NULL DEFAULT current_timestamp(),
    check_name      STRING    NOT NULL,
    model_name      STRING    NOT NULL,
    result_value    DOUBLE    COMMENT 'Value returned by the DAX for this model',
    baseline_model  STRING    COMMENT 'The first model registered for this check_name — used as reference',
    baseline_value  DOUBLE    COMMENT 'Value returned by the DAX for the baseline model',
    absolute_delta  DOUBLE    COMMENT 'result_value minus baseline_value',
    delta_pct       DOUBLE    COMMENT 'absolute_delta divided by baseline_value, as a percentage',
    status          STRING    NOT NULL COMMENT 'PASS | FAIL | ERROR',
    error_message   STRING    COMMENT 'Populated only when status = ERROR'
)
USING DELTA
PARTITIONED BY (run_date)
COMMENT 'Validation results written by the daily job — do not edit manually'
""")

print(f"✓  Tables ready in {FULL_SCHEMA}")

## Verify

In [ ]:
from IPython.display import display

# Show both tables exist
tables = spark.sql(f"SHOW TABLES IN {SCHEMA_NAME}").toPandas()
display(tables)

print(f"\n✓  Both tables created in {FULL_SCHEMA}")
print("   Next: go to data_quality_add_checks_notebook to add your checks")